# Train an image classifier with TIMM (GeoAI)

This notebook replicates the **GeoAI** walkthrough [*Train timm classifier*](https://opengeoai.org/examples/train_timm_classifier/), which shows how to fine-tune **PyTorch Image Models** ([timm](https://github.com/huggingface/pytorch-image-models)) on image classification—including remote-sensing style pipelines—using the high-level API in `geoai.timm_train`.

**Original tutorial:** [opengeoai.org/examples/train_timm_classifier/](https://opengeoai.org/examples/train_timm_classifier/)

## Index

**Setup**

1. [Key ideas](#key-ideas)
2. [Install packages](#install)
3. [Get the course repository](#clone-repo)
4. [Imports](#imports)

**Data**

5. [Explore available timm models](#model-zoo)
6. [Download sample data (EuroSAT RGB)](#dataset)
7. [Prepare file lists and labels](#file-lists)
8. [Show class distribution](#class-distribution)
9. [Train / validation / test splits](#splits)
10. [Visualize one example per class](#visualize-classes)
11. [Build `RemoteSensingDataset` objects](#datasets-transforms)

**Training**

12. [Train a ResNet-50 classifier](#train-resnet50)
13. [Train EfficientNet-B3](#train-efficientnet-b3)
14. [Third run: frozen backbone (feature extraction)](#train-frozen)

**Evaluation**

15. [Load the trained models and run inference](#inference)
16. [Visualize predictions on a batch](#visualize-predictions)
17. [Quantitative evaluation on the test split](#test-evaluation)
18. [First conclusion](#first-conclusion)

**Out of distribution**

19. [Out-of-distribution test: high-resolution aerial imagery](#ood-test)
20. [Evaluating model performance without ground truth](#ood-evaluation)
21. [What actually went wrong: domain shift, not class imbalance](#domain-shift)
22. [How to fix it](#how-to-fix)
23. [Generating ground truth for the dataset](#ground-truth)
24. [Key takeaways](#key-takeaways)
25. [Summary](#summary)

<a name='key-ideas'></a>
## Key ideas

- **Large model zoo:** timm exposes many pretrained backbones (ResNet, EfficientNet, ViT, ConvNeXt, …).
- **Multi-channel inputs:** You can set `in_channels` (and the dataset's channel count) for RGB, RGB+NIR, etc. (see `train_timm_classifier`).
- **Lightning training loop:** Checkpointing, CSV logs, and early stopping are handled by PyTorch Lightning.
- **Transfer learning:** Load ImageNet weights (`pretrained=True`) and optionally **freeze the backbone** to train only the head. Both are transfer learning; only updating the pretrained weights is *fine-tuning*.
- **Preprocess the way the checkpoint expects:** every timm checkpoint carries its own input size and mean/std. Read them from the model, never hardcode them.

`RemoteSensingDataset` in GeoAI loads rasters with **rasterio** (paths must be readable by your GDAL/rasterio build, including many JPEG/PNG setups).

<a name='install'></a>
## Install packages

Install GeoAI, timm, Lightning and Hugging Face `datasets` (for EuroSAT in this demo). Rasterio is pulled in as needed for reading image files.

We deliberately **do not** install `torch`, `torchvision`, `matplotlib` or `scikit-learn`: Colab already ships them with a CUDA-matched build, and replacing them with generic PyPI wheels is the quickest way to lose the GPU runtime.

In [ ]:
# Only the packages Colab lacks. `torch` / `torchvision` are intentionally absent.
!pip install -q geoai-py timm lightning datasets

# Why not `uv`? Outside a virtual environment `uv pip install` aborts with
#   "No virtual environment found; run `uv venv` ... or pass `--system`"
# and a Colab runtime is exactly that: the system interpreter, no venv. Because `!` never
# raises, the notebook would happily continue and die later on the first import.
# If you really want uv:  !pip install -q uv && uv pip install --system -q geoai-py timm lightning datasets

<a name='clone-repo'></a>
## Get the course repository

A Colab runtime is a bare virtual machine: it has no copy of this course. Two things in this lab
live in the repository and not on PyPI, so we clone it and make it the working directory:

- `helpers/timm_train.py` — the patched `train_timm_classifier` we import in the next cell;
- `media/datasets/timm/hq/` — the high-resolution aerial images used in the out-of-distribution
  section at the end.

The clone reads the **remote** `main` branch, so any file you add locally has to be committed and
pushed before Colab can see it.

In [ ]:
import os

REPO_URL = "https://github.com/EzequielMatiasArevalo/tuia-computer-vision.git"
REPO_DIR = "tuia-computer-vision"

# Safe to re-run: clone only if it is not already there, cd only if we are not already inside.
if not os.path.isdir(REPO_DIR) and os.path.basename(os.getcwd()) != REPO_DIR:
    !git clone -q $REPO_URL

if os.path.basename(os.getcwd()) != REPO_DIR:
    %cd $REPO_DIR

print("Working directory:", os.getcwd())
print("Helpers present:", os.path.isfile("helpers/timm_train.py"))
print("OOD images present:", os.path.isdir("media/datasets/timm/hq"))

<a name='imports'></a>
## Imports

In [ ]:
import os
import tempfile

from geoai.timm_train import (
    RemoteSensingDataset,
    TimmClassifier,
    list_timm_models,
    predict_with_timm,
)

# `train_timm_classifier` comes from the repository we cloned above, not from geoai:
# the upstream class has a bug in its logger call (line 523), patched in helpers/timm_train.py.
# This import only works because the previous cell made the repo root our working directory.
from helpers.timm_train import train_timm_classifier

<a name='model-zoo'></a>
## Explore available timm models

Filter names to browse architectures before picking `model_name` for training.

In [ ]:
resnet_models = list_timm_models(filter="resnet", limit=10)
print("ResNet models:", resnet_models)


In [ ]:
efficientnet_models = list_timm_models(filter="efficientnet", limit=10)
print("EfficientNet models:", efficientnet_models)


In [ ]:
vit_models = list_timm_models(filter="vit", limit=10)
print("Vision Transformer models:", vit_models)


<a name='dataset'></a>
## Download sample data (EuroSAT RGB)

The tutorial uses the **EuroSAT RGB** dataset from Hugging Face ([`timm/eurosat-rgb`](https://huggingface.co/datasets/timm/eurosat-rgb)): Sentinel-2 RGB chips of 64 × 64 pixels at **10 m/pixel** ground sampling distance, over **10** land-cover classes:

- AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake

Remember that 10 m/pixel figure — one chip covers 640 m × 640 m of ground. It is the number the whole final section of this notebook turns on.

Images are written to a temporary folder in **class subdirectories** (ImageFolder layout) so we can build path lists and integer labels. The dataset's three predefined splits are kept as they are.

In [ ]:
from datasets import load_dataset
from PIL import Image

print("Loading EuroSAT from Hugging Face...")
dataset = load_dataset("timm/eurosat-rgb")

train_temp_dir = tempfile.mkdtemp(prefix="eurosat_train")
print(f"Saving images to: {train_temp_dir}")

test_temp_dir = tempfile.mkdtemp(prefix="eurosat_test")
print(f"Saving images to: {test_temp_dir}")

val_temp_dir = tempfile.mkdtemp(prefix="eurosat_val")
print(f"Saving images to: {val_temp_dir}")

class_names = dataset["train"].features["label"].names

print("Classes:", class_names)

for dataset, temp_dir in zip([dataset["train"], dataset["test"], dataset["validation"]], [train_temp_dir, test_temp_dir, val_temp_dir]):
    for idx, sample in enumerate(dataset):
        img = sample["image"]
        label = sample["label"]
        class_name = class_names[label]
        class_dir = os.path.join(temp_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)
        img_path = os.path.join(class_dir, f"{idx:05d}.jpg")
        img.save(img_path)

print(f"Saved {len(dataset)} images to {temp_dir}")


<a name='file-lists'></a>
## Prepare file lists and labels

Collect paths in class order, assign **integer labels** `0 … num_classes-1`, then print class counts.

In [ ]:
import glob

image_paths = {
    "train": [],
    "test": [],
    "val": []
}

labels = {
    "train": [],
    "test": [],
    "val": []
}

for t,temp_dir in zip(["train", "test", "val"], [train_temp_dir, test_temp_dir, val_temp_dir]):
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(temp_dir, class_name)
        class_images = sorted(glob.glob(os.path.join(class_dir, "*.jpg")))
        image_paths[t].extend(class_images)
        labels[t].extend([class_idx] * len(class_images))

    print(f"Dataset: {t}")
    print(f"Total images: {len(image_paths[t])}")
    print(f"Number of classes: {len(class_names)}")
    print("Class distribution:")
    for class_idx, class_name in enumerate(class_names):
        count = labels[t].count(class_idx)
        print(f"  {class_name}: {count}")

<a name='class-distribution'></a>
## Show class distribution

Worth reading carefully: the spread between the largest and smallest class here is the number that
tells you whether "class imbalance" is a credible explanation for anything later in the notebook.

In [ ]:
print("Plot class distribution and balance")

import matplotlib.pyplot as plt
import numpy as np

# Count samples per class
for t in ["train", "test", "val"]:
    counts = [labels[t].count(i) for i in range(len(class_names))]

    plt.figure(figsize=(12, 6))
    bars = plt.bar(class_names, counts, color='skyblue')
    plt.title("Class Distribution in EuroSAT Dataset")
    plt.xlabel("Class")
    plt.ylabel("Number of images")
    plt.xticks(rotation=25, ha='right')

    # Annotate bars with counts
    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, str(count),
                ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()

    print("Mean images per class:", np.mean(counts))
    print("Standard deviation:", np.std(counts))

<a name='splits'></a>
## Train / validation / test splits

**We do not split anything here.** `timm/eurosat-rgb` ships three predefined splits —
`train` / `validation` / `test`, the canonical **60 / 20 / 20** partition of the 27,000 EuroSAT
chips — and the cells above already wrote each one to its own folder. Re-splitting a dataset that
has an official partition makes your numbers incomparable with everyone else's, so we keep it.

The cell below is kept as a **recipe for the other case**: a dataset of loose files with no
predefined splits. It is commented out on purpose. Two details worth copying from it:

- **stratify** on the labels, so every class keeps its proportion in each split;
- a fixed **`random_state`**, so the split is the same on every run — otherwise the "test" set
  quietly changes between sessions and leaks into training.

Note that a nested 80/20-of-80 split gives 64 / 16 / 20, not 60 / 20 / 20; that is one more reason
to prefer a dataset's official partition when it has one.

In [ ]:
# NOT RUN — reference recipe for datasets that have no predefined splits.
# `all_paths` / `all_labels` would be one flat list over every image.
#
# from sklearn.model_selection import train_test_split
#
# train_paths, test_paths, train_labels, test_labels = train_test_split(
#     all_paths,
#     all_labels,
#     test_size=0.2,
#     random_state=42,
#     stratify=all_labels,      # keep the class proportions
# )
#
# train_paths, val_paths, train_labels, val_labels = train_test_split(
#     train_paths,
#     train_labels,
#     test_size=0.2,            # 20% of the remaining 80% -> 64 / 16 / 20 overall
#     random_state=42,
#     stratify=train_labels,
# )
#
# print(f"Training samples: {len(train_paths)}")
# print(f"Validation samples: {len(val_paths)}")
# print(f"Test samples: {len(test_paths)}")

for split in ["train", "val", "test"]:
    print(f"{split:>5}: {len(image_paths[split]):>6} images "
          f"({100 * len(image_paths[split]) / sum(len(v) for v in image_paths.values()):.0f}%)")

<a name='visualize-classes'></a>
## Visualize one example per class

Quick sanity check of labels and appearance before training.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for idx, class_name in enumerate(class_names):
    ax = axes[idx // 5, idx % 5]
    img_idx = labels["train"].index(idx)
    img = Image.open(image_paths["train"][img_idx])
    ax.imshow(img)
    ax.set_title(class_name, fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()


<a name='datasets-transforms'></a>
## Build `RemoteSensingDataset` objects — with the preprocessing the backbone expects

`RemoteSensingDataset` reads the raster with rasterio, keeps the first `num_channels` bands and
divides by 255. That is **all** it does: the tensor it returns is `(3, 64, 64)` in raw `[0, 1]`.

That is not what our backbones were trained on:

| Checkpoint | Native input | Normalization |
| --- | --- | --- |
| `resnet50.a1_in1k` | 3 × 224 × 224 | ImageNet mean/std |
| `efficientnet_b3.ra2_in1k` | 3 × 288 × 288 | ImageNet mean/std |

Feeding 64 × 64 tensors in raw `[0, 1]` to an ImageNet backbone breaks the premise of the whole
lab. Two separate problems:

1. **No standardization.** Every pretrained filter was fitted on inputs with roughly zero mean and
   unit variance per channel. Handing it a `[0, 1]` input shifts the response of the very first
   convolution, and the error compounds layer by layer.
2. **No resize.** EfficientNet-B3's stem has an output stride of 32, so a 64 × 64 image comes out
   of the backbone as a **2 × 2** feature map before global pooling — almost nothing left to
   average over. The frozen-backbone run suffers most, because it cannot adapt to the mismatch at
   all: it has to classify from features computed the wrong way.

The fix is never to hardcode the numbers. **Read them from the model**, with
`timm.data.resolve_model_data_config`, and pass the resulting transform to the dataset. Because the
two architectures want different resolutions, we build the datasets **per model**.

> Note on the transform pipeline: `create_transform` from `timm.data` expects a **PIL image**,
> while `RemoteSensingDataset` hands us a float tensor. So we compose the equivalent tensor-side
> transform ourselves — resize, then normalize — from the same config dict.

In [ ]:
import timm
import torch
from torchvision import transforms as T

INTERPOLATIONS = {
    "nearest": T.InterpolationMode.NEAREST,
    "bilinear": T.InterpolationMode.BILINEAR,
    "bicubic": T.InterpolationMode.BICUBIC,
}


def timm_data_config(model_name):
    """Return the preprocessing the pretrained checkpoint was trained with.

    `pretrained=False` is enough: timm attaches `pretrained_cfg` at build time, so we get the
    input size / mean / std / interpolation without downloading any weights.
    """
    model = timm.create_model(model_name, pretrained=False)
    cfg = timm.data.resolve_model_data_config(model)
    del model
    return cfg


def build_transform(cfg, train):
    """Tensor-side equivalent of `timm.data.create_transform` for a CHW float tensor in [0, 1]."""
    size = tuple(cfg["input_size"][1:])
    interpolation = INTERPOLATIONS.get(cfg.get("interpolation", "bicubic"),
                                       T.InterpolationMode.BICUBIC)

    steps = [T.Resize(size, interpolation=interpolation, antialias=True)]
    if train:
        # Land cover has no canonical "up", so both flips are label-preserving here.
        steps += [T.RandomHorizontalFlip(), T.RandomVerticalFlip()]
    steps.append(T.Normalize(mean=list(cfg["mean"]), std=list(cfg["std"])))
    return T.Compose(steps)


def make_datasets(model_name):
    """Build train/val/test datasets preprocessed the way `model_name` expects."""
    cfg = timm_data_config(model_name)
    print(f"{model_name}: input_size={cfg['input_size']} "
          f"mean={tuple(round(m, 3) for m in cfg['mean'])} "
          f"std={tuple(round(s, 3) for s in cfg['std'])} "
          f"interpolation={cfg.get('interpolation')}")

    datasets = {
        split: RemoteSensingDataset(
            image_paths=image_paths[split],
            labels=labels[split],
            num_channels=3,
            transform=build_transform(cfg, train=(split == "train")),
        )
        for split in ["train", "val", "test"]
    }
    return datasets, cfg


# Sanity check on one architecture: what actually reaches the network?
_datasets, _cfg = make_datasets("resnet50")
_x, _y = _datasets["train"][0]
print(f"\nTensor into the model: shape={tuple(_x.shape)} "
      f"min={_x.min():.2f} max={_x.max():.2f} mean={_x.mean():.2f}")
print("Train / val / test sizes:", {k: len(v) for k, v in _datasets.items()})

<a name='train-resnet50'></a>
## Train a ResNet-50 classifier

Fine-tune **ResNet-50** with ImageNet weights on the 10 EuroSAT classes. Training runs for several epochs; reduce `num_epochs` for a quick smoke test. Note that the datasets are rebuilt here, so the chips are resized and normalized the way `resnet50` expects.

### `last.ckpt` is not the best checkpoint

`train_timm_classifier` sets up two callbacks that interact:

- `ModelCheckpoint(monitor="val_acc", mode="max", save_top_k=1, save_last=True)` — writes the
  best epoch as `resnet50_{epoch:02d}_{val_loss:.4f}.ckpt`, **and** the final epoch as `last.ckpt`;
- `EarlyStopping(patience=5)` — stops only after 5 epochs without improvement.

Put together: `last.ckpt` is by construction up to `patience` epochs *past* the best `val_acc` —
the very epochs early stopping judged to be getting worse. The file you want for inference is the
`*_epoch*.ckpt` one, or `checkpoint_callback.best_model_path` if you have the trainer at hand. We
resolve it explicitly in the inference section below.

**Checkpoints** land under `output_dir/models/`. Monitoring `val_acc` with `mode="max"` matches the tutorial's accuracy-focused early stopping.

In [ ]:
resnet_datasets, resnet_cfg = make_datasets("resnet50")

output_dir = "timm_output/resnet50"
if not os.path.exists(output_dir):
    model = train_timm_classifier(
        train_dataset=resnet_datasets["train"],
        val_dataset=resnet_datasets["val"],
        test_dataset=resnet_datasets["test"],
        model_name="resnet50",
        num_classes=len(class_names),
        in_channels=3,
        pretrained=True,
        output_dir=output_dir,
        batch_size=32,
        num_epochs=20,
        learning_rate=1e-3,
        weight_decay=1e-4,
        num_workers=4,
        freeze_backbone=False,
        monitor_metric="val_acc",
        mode="max",
        patience=5,
        save_top_k=1,
    )

<a name='train-efficientnet-b3'></a>
## Train EfficientNet-B3

A different point on the accuracy–compute curve: **EfficientNet-B3** has roughly **12M** parameters
against ResNet-50's **25.6M**, but it is not simply "the small one". Its checkpoint
(`efficientnet_b3.ra2_in1k`) was trained at **288 × 288**, larger than ResNet-50's 224 × 224 —
compound scaling raises depth, width *and* input resolution together — so a B3 forward pass is not
proportionally cheaper than its parameter count suggests.

`make_datasets` picks that 288 × 288 up automatically from the model config, which is exactly why
we build the datasets per model instead of once.

In [ ]:
effnet_datasets, effnet_cfg = make_datasets("efficientnet_b3")

output_dir = "timm_output/efficientnet_b3"
if not os.path.exists(output_dir):
    model = train_timm_classifier(
        train_dataset=effnet_datasets["train"],
        val_dataset=effnet_datasets["val"],
        test_dataset=effnet_datasets["test"],
        model_name="efficientnet_b3",
        num_classes=len(class_names),
        in_channels=3,
        pretrained=True,
        output_dir=output_dir,
        batch_size=32,
        num_epochs=20,
        learning_rate=1e-3,
        weight_decay=1e-4,
        num_workers=4,
        freeze_backbone=False,
        monitor_metric="val_acc",
        mode="max",
        patience=5,
        save_top_k=1,
    )

<a name='train-frozen'></a>
## Third run: frozen backbone (feature extraction)

Set `freeze_backbone=True` so only the **classification head** updates: faster, and less risk of
wrecking low-level filters on a small dataset. The GeoAI module freezes every parameter whose name
does not contain `fc`, `head` or `classifier`.

Terminology, because the two are constantly confused and this lab exists to contrast them:

- **Transfer learning** is the umbrella term — reusing weights learned on one task for another.
  All three runs in this notebook are transfer learning.
- **Fine-tuning** means the pretrained weights themselves are updated. That is the two runs above.
- **Feature extraction** (also *linear probing*) means the backbone is frozen and used as a fixed
  feature function; only the new head is trained. That is this run. No pretrained weight moves, so
  by definition nothing is being "tuned".

This run is also the one hurt most by getting preprocessing wrong: a fine-tuned backbone can partly
absorb a normalization mismatch during training, a frozen one cannot.

In [ ]:
output_dir = "timm_output/resnet50_frozen"
if not os.path.exists(output_dir):
    model_frozen = train_timm_classifier(
        train_dataset=resnet_datasets["train"],
        val_dataset=resnet_datasets["val"],
        test_dataset=resnet_datasets["test"],
        model_name="resnet50",
        num_classes=len(class_names),
        in_channels=3,
        pretrained=True,
        freeze_backbone=True,
        output_dir=output_dir,
        batch_size=32,
        num_epochs=10,
        learning_rate=1e-3,
        monitor_metric="val_acc",
        mode="max",
    )

<a name='inference'></a>
## Load the trained models and run inference

Two things we have to get right here, and both are easy to get wrong silently.

**1. Load the best checkpoint, not the last one.** As explained above, `last.ckpt` is the final
epoch, which early stopping reached *after* deciding things had stopped improving. We glob for the
`{model}_{epoch}_{val_loss}.ckpt` file that `ModelCheckpoint` wrote for the best `val_acc` instead.

**2. Preprocess at inference exactly as at training.** `predict_with_timm` builds its own
`RemoteSensingDataset` internally **without a transform**, so it would feed raw 64 × 64 `[0, 1]`
tensors to a network trained on normalized 224/288 input — the train/serve skew that quietly costs
accuracy in most first deployments. We define `predict_paths` below, which rebuilds the eval
transform from the model config and is otherwise the same loop.

Probabilities are always returned here, so we can use the softmax scores for error analysis and
calibration later.

In [ ]:
from datetime import datetime

from torch.utils.data import DataLoader

# Which timm architecture is behind each run directory (the frozen run is also a ResNet-50).
RUN_ARCH = {
    "resnet50": "resnet50",
    "efficientnet_b3": "efficientnet_b3",
    "resnet50_frozen": "resnet50",
}


def best_checkpoint(run_dir):
    """Return the best-`val_acc` checkpoint of a run, explicitly skipping `last.ckpt`."""
    models_dir = os.path.join(run_dir, "models")
    candidates = sorted(
        p for p in glob.glob(os.path.join(models_dir, "*.ckpt"))
        if os.path.basename(p) != "last.ckpt"
    )
    if not candidates:
        raise FileNotFoundError(
            f"No best checkpoint in {models_dir} — only `last.ckpt`? "
            "Check that training actually completed a validation epoch."
        )
    if len(candidates) > 1:
        print(f"  (found {len(candidates)} top-k checkpoints, using the last one)")
    return candidates[-1]


def predict_paths(model, arch, paths, batch_size=32, num_workers=2):
    """Predict on a list of image paths, applying the model's own eval preprocessing.

    This is `predict_with_timm` plus the transform it does not accept.
    """
    cfg = timm_data_config(arch)
    dataset = RemoteSensingDataset(
        image_paths=paths,
        labels=[0] * len(paths),          # dummy: inference only
        num_channels=3,
        transform=build_transform(cfg, train=False),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()

    all_preds, all_probs = [], []
    with torch.no_grad():
        for images, _ in loader:
            probs = torch.softmax(model(images.to(device)), dim=1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(probs.argmax(dim=1).cpu().numpy())

    return np.concatenate(all_preds), np.concatenate(all_probs)


run_dirs = [
    "timm_output/resnet50",
    "timm_output/efficientnet_b3",
    "timm_output/resnet50_frozen",
]

models = {}
for run_dir in run_dirs:
    run_name = os.path.basename(run_dir)
    checkpoint_path = best_checkpoint(run_dir)
    print(f"{run_name}: loading {os.path.basename(checkpoint_path)}")

    model = TimmClassifier.load_from_checkpoint(checkpoint_path)

    start_time = datetime.now()
    predictions, probabilities = predict_paths(
        model=model,
        arch=RUN_ARCH[run_name],
        paths=image_paths["val"][:20],
        batch_size=8,
    )
    inference_time = datetime.now() - start_time

    models[run_name] = {
        "predictions": predictions,
        "probabilities": probabilities,
        "model": model,
        "arch": RUN_ARCH[run_name],
        "checkpoint": checkpoint_path,
        "inference_time": inference_time,
    }

    print(f"  Predictions shape: {predictions.shape}")
    print(f"  Probabilities shape: {probabilities.shape}")
    print(f"  Sample predictions: {[class_names[p] for p in predictions[:5]]}")

for run_name, model_info in models.items():
    print("=" * 50)
    print(f"Model: {run_name}")
    print(f"Parameters: {sum(p.numel() for p in model_info['model'].parameters()):,}")
    print(f"Inference time (20 images): {model_info['inference_time']}")

In [ ]:
!du -shc timm_output/*

<a name='visualize-predictions'></a>
## Visualize predictions on a batch

`plot_predictions` colours each title by **correctness**: green when the predicted class matches the
ground-truth label, red when it does not. Confidence is the probability assigned to the argmax
class. Remember this rule — the out-of-distribution section later has no labels and has to colour by
something else.

Also note what this is *not*: twenty validation images are a sanity check, not an evaluation. The
measurement comes two sections down.

In [ ]:
def plot_predictions(predictions, probabilities, test_paths, test_labels, class_names):
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))

    for idx, ax in enumerate(axes.flat):
        if idx >= len(test_paths[:20]):
            break
        img = Image.open(test_paths[idx])
        ax.imshow(img)
        pred_class = class_names[predictions[idx]]
        true_class = class_names[test_labels[idx]]
        confidence = probabilities[idx][predictions[idx]] * 100
        color = "green" if predictions[idx] == test_labels[idx] else "red"
        ax.set_title(
            f"Pred: {pred_class}\nTrue: {true_class}\n({confidence:.1f}%)",
            color=color,
            fontsize=10,
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()


### Predictions of the three runs on the same 20 validation images

In order: fully fine-tuned ResNet-50, fully fine-tuned EfficientNet-B3, frozen-backbone ResNet-50.

In [ ]:
plot_predictions(models["resnet50"]["predictions"], models["resnet50"]["probabilities"], image_paths["val"], labels["val"], class_names)

In [ ]:
plot_predictions(models["efficientnet_b3"]["predictions"], models["efficientnet_b3"]["probabilities"], image_paths["val"], labels["val"], class_names)

In [ ]:
plot_predictions(models["resnet50_frozen"]["predictions"], models["resnet50_frozen"]["probabilities"], image_paths["val"], labels["val"], class_names)

<a name='test-evaluation'></a>
## Quantitative evaluation on the test split

Everything so far has been eyeballing twenty pictures. Twenty hand-picked images cannot tell you
whether one model is better than another, cannot tell you *which* classes fail, and — because those
twenty came from the **validation** split, the split early stopping already optimized against — they
are not even an honest estimate of generalization.

So before drawing any conclusion, we score all three runs over the **entire test split** and print:

- **accuracy** — one number per model, for the ranking;
- **`classification_report`** — per-class precision / recall / F1, which is where the interesting
  information is. Precision answers "when it says *River*, how often is it right?"; recall answers
  "of all the real *River* chips, how many did it find?". A class can have high precision and
  terrible recall, and accuracy hides that completely;
- **macro vs weighted average** — macro treats every class equally, weighted weights by support.
  On a roughly balanced dataset like EuroSAT they stay close; when they diverge, the gap is the
  imbalance;
- a **confusion matrix**, which tells you *what* a class is mistaken for. Confusions between
  `Highway` / `River` or `PermanentCrop` / `AnnualCrop` are the ones to look for — they are pairs
  that genuinely look alike at 10 m/pixel.

This takes a few minutes: it is three models over ~5,400 chips.

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

test_labels = np.asarray(labels["test"])

for run_name, model_info in models.items():
    print("=" * 70)
    print(f"{run_name}  —  {os.path.basename(model_info['checkpoint'])}")
    print("=" * 70)

    test_preds, test_probs = predict_paths(
        model=model_info["model"],
        arch=model_info["arch"],
        paths=image_paths["test"],
        batch_size=64,
    )
    model_info["test_predictions"] = test_preds
    model_info["test_probabilities"] = test_probs

    accuracy = accuracy_score(test_labels, test_preds)
    print(f"Test accuracy: {accuracy:.4f}  ({len(test_labels)} images)\n")

    print(classification_report(
        test_labels,
        test_preds,
        target_names=class_names,
        digits=3,
        zero_division=0,
    ))

    model_info["test_accuracy"] = accuracy

# Side-by-side ranking
print("\nTest accuracy, all runs")
for run_name, model_info in sorted(models.items(),
                                   key=lambda kv: kv[1]["test_accuracy"],
                                   reverse=True):
    print(f"  {run_name:<20} {model_info['test_accuracy']:.4f}")

In [ ]:
# Row-normalized confusion matrices: each row sums to 1, so the diagonal is per-class recall
# and every off-diagonal cell reads as "x% of this class was called that instead".
fig, axes = plt.subplots(1, len(models), figsize=(7 * len(models), 6.5))
axes = np.atleast_1d(axes)

for ax, (run_name, model_info) in zip(axes, models.items()):
    cm = confusion_matrix(
        test_labels,
        model_info["test_predictions"],
        labels=range(len(class_names)),
        normalize="true",
    )
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=90
    )
    ax.set_title(f"{run_name}\ntest accuracy = {model_info['test_accuracy']:.3f}")

plt.tight_layout()
plt.show()

# The three classes each model recovers worst — this is what "which classes fail" looks like
# when it is measured instead of guessed from twenty thumbnails.
for run_name, model_info in models.items():
    cm = confusion_matrix(test_labels, model_info["test_predictions"],
                          labels=range(len(class_names)), normalize="true")
    per_class_recall = cm.diagonal()
    worst = np.argsort(per_class_recall)[:3]
    print(f"{run_name}: lowest recall -> " +
          ", ".join(f"{class_names[i]} ({per_class_recall[i]:.2f})" for i in worst))

<a name='first-conclusion'></a>
## First conclusion

We trained three models, and all three are **transfer learning** from ImageNet weights — that is the
umbrella term, not one of the three strategies. What differs is how much of the pretrained network
we allowed to move:

| Run | Backbone | Strategy | What was updated |
| --- | --- | --- | --- |
| `resnet50` | ResNet-50 | **Full fine-tuning** | every weight |
| `efficientnet_b3` | EfficientNet-B3 | **Full fine-tuning** | every weight |
| `resnet50_frozen` | ResNet-50 | **Feature extraction** (linear probing) | the classification head only |

The distinction matters and is routinely stated backwards: freezing the backbone is *not*
fine-tuning, because no pretrained weight is tuned — the backbone is used as a fixed feature
function and only the new head learns. Fine-tuning is the other two runs.

The test-split numbers above tell you how they rank on EuroSAT, per class and overall.

> **Exercise.** The model zoo we listed at the top also has `convnext_tiny`. Add a fourth run with
> it — same `make_datasets("convnext_tiny")`, same call — and you get a three-way architectural
> comparison: a classic CNN, an efficiency-optimized CNN and a modern ConvNet, all under one
> training recipe. Predict the ranking before you run it.

*So far, so good — on data drawn from the same distribution as training. But what happens when we
run inference on external images from outside that distribution? How do factors such as image
resolution, sensor, altitude and perspective affect the model?*

<a name='ood-test'></a>
## Out-of-distribution test: high-resolution aerial imagery

The images below come from the course repository (`media/datasets/timm/hq/`) and are **not**
EuroSAT. They are high-resolution aerial photographs, and the difference that matters is not
"quality" — it is **ground sampling distance** (GSD), the size of the patch of Earth that one pixel
covers.

| | EuroSAT | Our aerial images |
| --- | --- | --- |
| Sensor | Sentinel-2, from orbit | aerial / sub-metre |
| GSD | **10 m/pixel** | roughly **0.3 m/pixel** |
| What a 64 × 64 chip covers | **640 m × 640 m** | roughly **20 m × 20 m** |

Same tensor shape, ground footprints ~30× apart in each dimension. A 64 × 64 EuroSAT chip of
`Residential` is a whole neighbourhood — a texture of roof-sized blobs. A 64 × 64 crop of the
aerial image is a fragment of one roof, or a piece of a single car. The model has never seen either.

Keep this in mind as you read the predictions: whatever goes wrong below, the first suspect is the
scale mismatch, not the model.

In [ ]:
# Path inside the cloned repository (note: `datasets`, plural).
satellite_custom_images = "media/datasets/timm/hq"
temp_path = "/tmp/timm"

os.makedirs(temp_path, exist_ok=True)

if not os.path.isdir(satellite_custom_images):
    raise FileNotFoundError(
        f"{satellite_custom_images} not found. It ships with the repository cloned at the top of "
        "the notebook — if you added the images locally, commit and push them to `main` first."
    )

images_path = [p for p in sorted(os.listdir(satellite_custom_images)) if not p.startswith(".")]
print(f"Ploting original {len(images_path)} images...")

n_axes = 2
n_rows_calc = int(len(images_path)/n_axes)
n_rows = 1 if n_rows_calc == 0 else n_rows_calc
fig, axes = plt.subplots(n_rows, n_axes, figsize=(20, 16))
axes = np.atleast_1d(axes)
idx = 0

for image_path in images_path:
    image = Image.open(f"{satellite_custom_images}/{image_path}")

    if n_rows > 1:
        ax = axes[idx // n_axes, idx % n_axes]
    else:
        ax = axes[idx]
    idx += 1
    ax.imshow(image)
    ax.set_title(
        f"{image_path} — {image.size[0]}x{image.size[1]} px",
        fontsize=10,
    )
    ax.axis("off")

In [ ]:
# We cut 64x64 crops so the tensor shape matches EuroSAT exactly.
# Shape is all that matches: at ~0.3 m/pixel these crops cover ~20 m of ground,
# where a 64x64 EuroSAT chip covers 640 m.
print("Cropping images in batches of 64x64 pixels...")
def crop_image_batch(image, size=(64, 64)):

    width, height = image.size
    for i in range(0, width, size[0]):
        for j in range(0, height, size[1]):
            yield image.crop((i, j, i + size[0], j + size[1]))

cropped_images = {}
for image_path in images_path:
    image_name = image_path.split(".")[0]
    image = Image.open(f"{satellite_custom_images}/{image_path}")
    cropped_images[image_name] = list(crop_image_batch(image))
    print(f"Cropped {image_name} {len(cropped_images[image_name])} images...")

In [ ]:

import random

cropped_path = f"{temp_path}/cropped-hq"
os.makedirs(cropped_path, exist_ok=True)

for image_name, imgs in cropped_images.items():
    n_axes = 5
    n_rows = 3
    fig, axes = plt.subplots(n_rows, n_axes, figsize=(20, 16))

    plt.title(f"{image_name}", fontsize=10)
    random.shuffle(imgs)

    for idx, ax in enumerate(axes.flat):
        if idx >= len(imgs):
            break
        ax.imshow(imgs[idx])
        #ax.axis("off")
        imgs[idx] = imgs[idx].convert("RGB")
        imgs[idx].save(f"{cropped_path}/{image_name}_{idx}.png")

    plt.tight_layout()
print(f"Cropped path: {cropped_path}")

### The colour code changes here — read this before the plots

These crops have **no ground-truth labels**, so the green/red rule used earlier ("green = the
prediction matches the label") cannot be evaluated. The function below is a *different* function —
`plot_predictions_no_gt` — and it colours by **confidence** instead: green above
`CONFIDENCE_THRESHOLD`, red below.

That is a much weaker signal, and the trap is worth naming explicitly: **a confidently wrong
prediction renders green.** In this section, where the whole point is that the model is out of its
depth, green means only "the softmax is peaked", never "correct". A network that has never seen
20-metre-wide crops can be, and often is, serenely confident about the wrong class.

We keep the two functions under two different names so a green title never silently changes meaning.

In [ ]:
# Green above this softmax probability, red below. Green means "confident", NOT "correct".
CONFIDENCE_THRESHOLD = 50.0  # percent


def plot_predictions_no_gt(predictions, probabilities, images_path, class_names,
                           threshold=CONFIDENCE_THRESHOLD, title=None):
    """Plot predictions for unlabelled images, coloured by confidence (no ground truth here)."""
    n_axes = 5
    n_rows = max(1, len(images_path) // n_axes)
    fig, axes = plt.subplots(n_rows, n_axes, figsize=(20, 16))
    if title:
        fig.suptitle(f"{title} — green = confidence > {threshold:.0f}% (not accuracy)", fontsize=14)

    for idx, ax in enumerate(np.atleast_1d(axes).flat):
        if idx >= len(images_path):
            ax.axis("off")
            continue
        image = Image.open(images_path[idx])
        ax.imshow(image)
        pred_class = class_names[predictions[idx]]
        confidence = probabilities[idx][predictions[idx]] * 100
        color = "green" if confidence > threshold else "red"
        ax.set_title(
            f"Pred: {pred_class}\n ({confidence:.1f}%)",
            color=color,
            fontsize=10,
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
print("Predicting Resnet50...")

cropped_images_paths = sorted(
    f"{cropped_path}/{image_path}" for image_path in os.listdir(cropped_path)
)

ood_predictions, ood_probabilities = predict_paths(
    model=models["resnet50"]["model"],
    arch=models["resnet50"]["arch"],
    paths=cropped_images_paths,
    batch_size=8,
)

print(f"Predictions: {ood_predictions}")
print("Mean confidence: "
      f"{100 * ood_probabilities.max(axis=1).mean():.1f}%  "
      "(high confidence on data the model has never seen is a warning sign, not a result)")

In [ ]:
plot_predictions_no_gt(ood_predictions, ood_probabilities, cropped_images_paths, class_names,
                       title="ResNet-50 on ~0.3 m/pixel crops")

In [ ]:
print("Predicting efficientnet_b3...")

eff_ood_predictions, eff_ood_probabilities = predict_paths(
    model=models["efficientnet_b3"]["model"],
    arch=models["efficientnet_b3"]["arch"],
    paths=cropped_images_paths,
    batch_size=8,
)

print(f"Predictions: {eff_ood_predictions}")

plot_predictions_no_gt(eff_ood_predictions, eff_ood_probabilities, cropped_images_paths,
                       class_names, title="EfficientNet-B3 on ~0.3 m/pixel crops")

### Testing the hypothesis: is it the scale?

Blaming a failure is cheap; the useful move is to design the experiment that isolates the cause.
If the problem really is ground sampling distance, then giving the model the *same* aerial
photograph at the *same* tensor shape but at EuroSAT's ground footprint should change the
predictions. Nothing else about the data changes — same sensor, same scene, same colours, same
64 × 64 input.

The mechanism: instead of cutting 64 × 64 **pixel** crops, we cut 64·f × 64·f pixel windows and
resize each one down to 64 × 64. That multiplies the effective GSD of the chip by `f` while keeping
the input shape fixed. At `f = 32` and an assumed source GSD of ≈0.3 m/pixel, each chip covers
≈9.6 m/pixel and ≈614 m of ground — EuroSAT's regime.

Read the output as a trend, not as a single number: as `f` grows, watch whether the predicted
classes stop being scattered and start being plausible land cover for a city beside a lake, and
whether confidence stops being uniformly high on nonsense.

One honest limitation, visible in the chip counts below: this photograph covers roughly one
kilometre of ground, so at the coarsest scale the whole image is worth only one or two EuroSAT-sized
chips. That is not a bug in the experiment — it is the 30× scale gap, stated in units of "how much
Earth do I even have".

In [ ]:
from collections import Counter

ASSUMED_SOURCE_GSD_M = 0.30   # metres per pixel of the aerial photograph (approximate)
EUROSAT_GSD_M = 10.0          # metres per pixel of Sentinel-2 RGB, i.e. of every training chip
CHIP_PX = 64
SCALE_FACTORS = [1, 4, 16, 32]
MAX_CHIPS = 30


def chips_at_scale(image, factor, out=CHIP_PX, max_chips=MAX_CHIPS):
    """Cut (out*factor)^2 windows and resize them to out^2: same shape, factor x coarser GSD."""
    window = out * factor
    window = min(window, min(image.size))          # cannot ask for more ground than we have
    stride = max(window // 2, 1)

    w, h = image.size
    coords = [(i, j)
              for i in range(0, w - window + 1, stride)
              for j in range(0, h - window + 1, stride)]
    if len(coords) > max_chips:                    # even subsample, not just the top-left corner
        coords = coords[:: max(1, len(coords) // max_chips)][:max_chips]

    chips = [
        image.crop((i, j, i + window, j + window)).resize((out, out), Image.BILINEAR).convert("RGB")
        for i, j in coords
    ]
    return chips, window


source_image = Image.open(f"{satellite_custom_images}/{images_path[0]}")
print(f"Source: {images_path[0]} {source_image.size[0]}x{source_image.size[1]} px "
      f"~= {source_image.size[0] * ASSUMED_SOURCE_GSD_M:.0f} x "
      f"{source_image.size[1] * ASSUMED_SOURCE_GSD_M:.0f} m of ground\n")

scale_results = {}
for factor in SCALE_FACTORS:
    chips, window = chips_at_scale(source_image, factor)
    effective_gsd = ASSUMED_SOURCE_GSD_M * window / CHIP_PX

    scale_dir = f"{temp_path}/scale_{factor:02d}"
    os.makedirs(scale_dir, exist_ok=True)
    chip_paths = []
    for k, chip in enumerate(chips):
        p = f"{scale_dir}/chip_{k:03d}.png"
        chip.save(p)
        chip_paths.append(p)

    preds, probs = predict_paths(
        model=models["resnet50"]["model"],
        arch=models["resnet50"]["arch"],
        paths=chip_paths,
        batch_size=8,
    )

    scale_results[factor] = {
        "paths": chip_paths,
        "predictions": preds,
        "probabilities": probs,
        "window_px": window,
        "effective_gsd_m": effective_gsd,
        "ground_m": window * ASSUMED_SOURCE_GSD_M,
        "mean_confidence": float(probs.max(axis=1).mean()),
        "distinct_classes": len(set(preds.tolist())),
    }

    counts = Counter(class_names[p] for p in preds)
    print(f"factor x{factor:<3} window={window:>4}px  "
          f"effective GSD ~{effective_gsd:5.2f} m/px  "
          f"ground/chip ~{window * ASSUMED_SOURCE_GSD_M:6.0f} m  "
          f"chips={len(chip_paths):>2}")
    print(f"    mean confidence {100 * probs.max(axis=1).mean():5.1f}%   "
          f"predicted: {dict(counts.most_common())}")
    print(f"    (EuroSAT reference: {EUROSAT_GSD_M:.0f} m/px, "
          f"{EUROSAT_GSD_M * CHIP_PX:.0f} m of ground per chip)\n")

In [ ]:
# Look at the chips themselves: one row per scale, so you can see what the model is being asked to
# classify. The bottom row is the only one that resembles a EuroSAT training chip.
n_show = 5
fig, axes = plt.subplots(len(SCALE_FACTORS), n_show,
                         figsize=(4 * n_show, 4 * len(SCALE_FACTORS)))
axes = np.atleast_2d(axes)

for row, factor in enumerate(SCALE_FACTORS):
    result = scale_results[factor]
    for col in range(n_show):
        ax = axes[row, col]
        ax.axis("off")
        if col >= len(result["paths"]):
            continue
        ax.imshow(Image.open(result["paths"][col]))
        pred = class_names[result["predictions"][col]]
        conf = result["probabilities"][col][result["predictions"][col]] * 100
        ax.set_title(f"x{factor} (~{result['effective_gsd_m']:.1f} m/px)\n{pred} ({conf:.0f}%)",
                     fontsize=10)

plt.tight_layout()
plt.show()

# And the trend, as two numbers per scale.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
factors = list(scale_results)
gsds = [scale_results[f]["effective_gsd_m"] for f in factors]

ax1.plot(gsds, [100 * scale_results[f]["mean_confidence"] for f in factors], "o-")
ax1.axvline(EUROSAT_GSD_M, ls="--", c="grey")
ax1.text(EUROSAT_GSD_M, ax1.get_ylim()[0], " EuroSAT GSD", color="grey", va="bottom")
ax1.set_xscale("log")
ax1.set_xlabel("effective GSD of the chip (m/pixel, log scale)")
ax1.set_ylabel("mean top-1 confidence (%)")
ax1.set_title("Confidence vs. ground sampling distance")

ax2.plot(gsds, [scale_results[f]["distinct_classes"] for f in factors], "o-")
ax2.axvline(EUROSAT_GSD_M, ls="--", c="grey")
ax2.set_xscale("log")
ax2.set_xlabel("effective GSD of the chip (m/pixel, log scale)")
ax2.set_ylabel("distinct classes predicted")
ax2.set_title("Prediction scatter vs. ground sampling distance")

plt.tight_layout()
plt.show()

<a name='ood-evaluation'></a>
## Evaluating model performance without ground truth

Is it possible to assess a model's performance without labels?

Only partly, and it is important to be precise about which part. **Quantitative** evaluation is
impossible — there is no accuracy, no per-class recall, no confusion matrix without a reference to
compare against. That is exactly why the test-split evaluation earlier in this notebook exists, and
why "twenty images looked fine" was never an acceptable substitute for it.

What remains available is **qualitative** analysis plus a few label-free signals:

* **Prediction distribution.** If a scene of a city beside a lake produces ten different classes in
  thirty adjacent chips, something is wrong regardless of what the right answer is.
* **Confidence, read with suspicion.** Softmax confidence is *not* a probability of correctness.
  Neural networks are famously overconfident off-distribution: a network trained on ten land-cover
  classes has no "none of the above" output, so it must spread all its mass over the ten it knows.
  High confidence on out-of-distribution input is a symptom, not a reassurance.
* **Spatial consistency.** Neighbouring chips of the same land cover should mostly agree. Disagreement
  between adjacent chips is a cheap, label-free error detector.
* **Agreement between models.** Our three runs are independent enough that a chip where they disagree
  is a chip to look at.

---

<a name='domain-shift'></a>
## What actually went wrong: domain shift, not class imbalance

The tempting diagnosis is class imbalance — "the model is bad at highways and rivers because it saw
fewer of them". Two things rule that out here:

1. **The data is nearly balanced.** The class-distribution plot at the top of this notebook shows
   roughly 3,000 images for the largest class against 2,000 for the smallest: a **1.5× spread**.
   Imbalance at that level shifts accuracy by a few points. It does not produce the collapse we saw.
2. **The test-split report shows the same models doing fine on those same classes** when the input
   comes from Sentinel-2. If `River` were fundamentally under-learned, it would be weak everywhere.

The real cause is **domain shift**, and specifically a **resolution / ground-sampling-distance
shift**:

| | EuroSAT (training) | Our aerial images (inference) |
| --- | --- | --- |
| Sensor | Sentinel-2, multispectral, orbital | aerial camera, RGB |
| GSD | 10 m/pixel | ≈0.3 m/pixel |
| Ground covered by a 64 × 64 chip | 640 m × 640 m | ≈20 m × 20 m |
| A `Residential` chip therefore shows | a whole neighbourhood as texture | part of one roof |
| A `River` chip therefore shows | a river crossing the frame | a patch of water with no banks |

That is a **30× scale mismatch in each dimension, ≈1000× in area**. A CNN's features are not
scale-invariant: the filters that fire on "the periodicity of a suburb at 10 m/pixel" have no
counterpart at 0.3 m/pixel, because at that resolution the periodicity is roof tiles. The scale
experiment above is the proof — the only thing changed between the runs was the ground footprint of
the chip, and the predictions moved.

Sensor differences compound it: Sentinel-2 RGB is calibrated top-of-atmosphere reflectance with its
own colour response, while an aerial photograph is whatever the camera's processing pipeline
produced. Different radiometry, different colour statistics, same nominal "RGB".

**The transferable lesson: in remote sensing, resolution and sensor domain shift is the dominant
failure mode.** Before reaching for a fancier loss, always ask first whether the inference data is
even in the same spatial regime as the training data.

---

<a name='how-to-fix'></a>
## How to fix it

### 1. Match the resolution (do this first)

* **Resample the inference imagery** to the training GSD, as the experiment above does. Cheap,
  immediate, and often enough.
* Or **retrain at the target resolution**, on a dataset that matches your sensor. For sub-metre
  aerial imagery that means datasets like UC Merced (0.3 m), AID or RESISC45, not EuroSAT.
* **Record the GSD in your metadata** and refuse to run when it is out of range. A model that says
  "out of domain" beats a model that says `River, 97%`.

### 2. Data-centric improvements

* **Multi-GSD training.** Augment by resampling training chips across a range of ground sampling
  distances, so the model sees each class at several scales. Note that this is *not* the "multi-scale
  training" of object detection, which is about objects of different sizes inside one image — here
  every chip has a single whole-image label, and what varies is how much ground the chip covers.
* Apply **radiometric augmentation** (brightness, contrast, colour jitter, mild blur and noise) to
  reduce sensitivity to the sensor's colour response.
* **Increase dataset diversity** for the classes that genuinely underperform on the *test split* —
  which you now know, because you measured it, rather than guessed it from twenty images.

### 3. Label quality and sampling

* Ensure annotations are **accurate and consistent**.
* Use **hard example mining** to focus training on difficult samples.
* If a real imbalance does show up in the per-class report, oversample the rare classes or use a
  **class-weighted loss** (`train_timm_classifier` accepts `class_weights`).

### 4. Model and training strategy

* Fine-tune deeper layers instead of freezing the backbone — the frozen run cannot adapt to a domain
  shift at all.
* If you do have a real imbalance, **Focal Loss** is worth trying. It reweights standard
  cross-entropy by `(1 − p_t)^γ`, where `p_t` is the probability the model assigned to the correct
  class: examples it already classifies confidently (`p_t → 1`) contribute almost nothing, so the
  gradient concentrates on the hard, usually rare, ones. `γ = 0` recovers plain cross-entropy;
  `γ = 2` is the usual default. It was introduced for one-stage detectors, where the
  background/foreground imbalance is thousands to one — a much more extreme regime than the 1.5×
  spread here, which is why it is the *last* thing to reach for in this lab, not the first.
* Add an **out-of-distribution check** (e.g. maximum softmax probability, or the distance to the
  training feature distribution) so the system can abstain instead of guessing.

---

<a name='ground-truth'></a>
## Generating ground truth for the dataset

If ground truth is not available, it must be created or approximated:

### 1. Manual annotation

* Use tools such as LabelImg, CVAT or Label Studio.
* Most accurate approach, but time-consuming.

### 2. Semi-automatic labeling

* Use the current model to generate **pseudo-labels**.
* Manually review and correct predictions.
* Iteratively retrain the model (active learning loop).
* Only sound when the model is *in* domain — pseudo-labelling across a 30× scale gap propagates the
  failure into the labels.

### 3. External data sources

* Leverage existing labeled datasets, choosing one that matches your **GSD and sensor**, not just
  your class list.
* Align and adapt them to your domain.

### 4. Weak supervision

* Use heuristic rules or metadata (e.g. OpenStreetMap overlays, GIS layers, cadastral data).
* Combine multiple weak signals to approximate labels.

---

<a name='key-takeaways'></a>
## Key takeaways

* **Measure before you conclude.** The test-split accuracy, per-class report and confusion matrix
  are the evidence; twenty thumbnails are an anecdote.
* Without ground truth, evaluation is qualitative and can reveal failure modes, but it cannot rank
  models — and **confidence is not correctness**, least of all off-distribution.
* In remote sensing, the dominant failure mode is **domain shift**, and its most common form is a
  mismatch in **ground sampling distance**. Check resolution before you blame the model.
* Diagnose with an experiment. Changing one variable — here, the ground footprint of the chip — and
  observing the predictions move is what turns "probably the scale" into a result.

<a name='summary'></a>
## Summary

1. **Clone the course repository** so Colab can see `helpers/` and `media/datasets/`, and install
   only the packages Colab lacks — never `torch`/`torchvision`.
2. **Choose a model** with `list_timm_models` or the timm docs.
3. **Prepare paths and integer labels**, keeping the dataset's official splits when it has them
   (here, EuroSAT's canonical 60/20/20 via Hugging Face → temp folders).
4. **Read the preprocessing from the model**, with `timm.data.resolve_model_data_config`, and wrap
   the data in `RemoteSensingDataset` with that transform and the correct channel count. Never
   hardcode mean/std or input size.
5. **Train** with `train_timm_classifier` — full fine-tuning, frozen backbone (feature extraction),
   optional `class_weights`. All of these are transfer learning; only the first is fine-tuning.
6. **Infer from the best checkpoint**, not `last.ckpt`, and apply the same eval transform used at
   training time.
7. **Evaluate quantitatively** on the *test* split: accuracy, per-class precision/recall/F1, and a
   confusion matrix. Twenty thumbnails are not an evaluation.
8. **Test out of distribution** and diagnose failures with an experiment. In remote sensing the
   first suspect is ground sampling distance, not class imbalance.